---
# <div style="text-align: center">Comparison of Molecular Species</div>
---

This notebook imports a random selection of cell2mol unit cells into SCOPE and compares their ligands. One representative of every unique ligand is retained, and ligands occurring in different unit cells are identified. The resulting ligand collection can then be used to prepare a common set of quantum-chemical calculations.

# Part 0: Configuration

The cell2mol dataset is stored outside the repository. Set `datasets_folder` to its parent folder. The expected structure is:

```text
<datasets_folder>/
└── cell2mol/
    ├── 1-Iron/
    ├── 2-Manganese/
    ├── 3-Ruthenium/
    ├── 4-Rhenium/
    ├── 5-Chromium/
    ├── 6-Cobalt/
    ├── 7-Nickel/
    └── 8-Copper/
```

The benchmark is skipped when the dataset folder is unavailable.

# Configuration

In [ ]:
import os
import sys

import scope
from rdkit import rdBase

sys.path.insert(0, os.path.abspath('..'))
from benchmark_functions import select_cell_paths

# Datasets are not uploaded to the repository. Set their parent folder here.
benchmark_folder        = os.path.abspath('./')
datasets_folder         = os.path.abspath('/Volumes/Science/Datasets')
cell2mol_dataset_folder = os.path.join(datasets_folder, 'cell2mol')

# Selected inputs, retained ligand objects, and benchmark reports are kept with the notebook.
cell_selection_file        = os.path.join(benchmark_folder, 'selected_cell_paths.txt')

## Set the number of Cells to be evaluated.
ncells = 100

## Existing selections and comparison results are reused unless their overwrite option is True.
overwrite_cell_selection    = False
comparison_debug            = 0

# Part 1: Collect unique ligands

## 1. Select cell2mol Cell objects

In [ ]:
## Reuse the existing Cell selection unless overwrite is requested.
cell2mol_dataset_available = os.path.isdir(cell2mol_dataset_folder)
relative_cell_paths        = []
cell_paths                 = []

if not cell2mol_dataset_available:
    print(f'Skipping ligand comparison: Cell dataset not found at {cell2mol_dataset_folder}')
elif os.path.isfile(cell_selection_file) and not overwrite_cell_selection:
    with open(cell_selection_file, 'r', encoding='utf-8') as selected_file:
        relative_cell_paths = [line.strip() for line in selected_file if line.strip()]
    print('Loaded existing Cell path selection')
else:
    relative_cell_paths    = select_cell_paths(ncells, cell2mol_dataset_folder)
    with open(cell_selection_file, 'w', encoding='utf-8') as selected_file:
        selected_file.write('\n'.join(relative_cell_paths))
    print('Created new Cell path selection')

if cell2mol_dataset_available:
    cell_paths = [os.path.join(cell2mol_dataset_folder, path) for path in relative_cell_paths]
    assert len(cell_paths) == ncells, f'Expected {ncells} paths; found {len(cell_paths)}'
    assert len(set(cell_paths)) == ncells, 'The selected path list contains repeated entries'
    assert all(os.path.isfile(path) for path in cell_paths), 'At least one selected Cell file is missing'
    print('Selected Cell files:', len(cell_paths))

Loaded existing Cell path selection
Selected Cell files: 100


## 2. Import Cells and compare ligands

Ligand identity is evaluated with SCOPE's existing `Ligand.__eq__()` implementation, and considers composition, connectivity, and graph topology. These elements suffice to distinguish up to tautomers, but does not distinguish conformers. 

In [3]:
from scope.classes_cell import import_cell

imported_cells = []
for index, cell_path in enumerate(cell_paths):
    cell2mol_cell = scope.load_binary(cell_path)
    with rdBase.BlockLogs():
        imported_cell = import_cell(cell2mol_cell)
        imported_cells.append(imported_cell)

In [4]:
verbose = False

unique_ligands = []
for cell in imported_cells:
    for mol in cell.molecules:
        if mol.iscomplex: 
            for lig in mol.ligands:  
                found = False
                for ulig in unique_ligands:
                    if lig == ulig[0]:
                        if verbose: print(f'Ligand {lig.formula} in {cell.name} also in {ulig[1]}')
                        found = True
                        break
                if not found:
                    unique_ligands.append([lig, cell.name])
print(len(unique_ligands), 'unique ligands found in', len(imported_cells), 'Cells')

138 unique ligands found in 100 Cells


## 3. Creates single System with all unique ligands

To take advantage of SCOPE automation and HPC handling, entities need to be grouped into a System (see Tutorial 1). One option is to create one system for each ligand. Another, more efficient, is to create a single system will all ligands as sources. Here, we follow that latter option.

To save computational time, only ligands with between 10 and 25 are included.

In [6]:
from scope.classes_system import System
sys = System("unique")

count = 1
for idx, (lig, name) in enumerate(unique_ligands):
    if lig.natoms >= 10 and lig.natoms <= 25:
        lig.name = "ulig_" + str(count)
        sys.add_source(lig.name, lig)    
        count += 1

# Important. Save the system to a valid path
sys.save(benchmark_folder+"/unique.npy") 

# Part 2: Prepare and Run SCOPE 

At this point, the System `unique` contains unique SCOPE Ligand objects for every ligand found in Part 1. At this stage, a user can follow the steps in his/her own HPC cluster with a SCOPE installation, and with Quantum Espresso version 7.0:

In [7]:
# Step 1. Set up the folder structure in the HPC cluster. Please have a look at Scheme 1 in Tutorial 6 for the expected structure
#         Hereafter, we will assume you work inside the Main project folder (black in scheme), called = $SCOPE_MAIN
# Step 2. If you don't have a valid SCOPE environment yet, run 'scope config -n bench' from $SCOPE_MAIN and follow instructions. 
#         This will create an environment file called scope_env_bench.npy.
#         During this process, you will be asked about folder paths. You can just set ./1-Sources, ./2-Systems, ./3-Computations
# Step 3. Upload the System file to the HPC cluster, in a dedicated folder inside the Systems folder.
#         ... for instance, in: $SCOPE_MAIN/2-Systems/unique/unique.npy 
# Step 4. To run QE computations with SCOPE, you might have to download the download the library of pseudopotentials. 
#         See SCOPE's README for information
# Step 5. Copy task1.scope and task2.scope to the HPC cluster, inside $SCOPE_MAIN 
# Step 6. Run 'scope run -n scope_env_bench.npy -s unique -i task1.scope task2.scope'
#         This steps must be repeated until all tasks are completed. 
# Step 7. Once all tasks are commpleted, download the System file to your local machine, in the same folder as this notebook.
#         To avoid overwriting the original System file, you can rename it to unique_finished.npy 

# Part 3: Prepare and Run SCOPE 

In [ ]:
# Loads the System file after HPC execution (see steps above)
sys = scope.load_binary(benchmark_folder+"/unique_finished.npy")

In [9]:
# Notice, in task1.scope and task2.scope, that Jobs were submitted under the branch called "bench".
found, branch = sys.find_branch("bench")
assert found, "Branch 'bench' not found in the system"

print(branch)

---------------------------------------------------
   >>> BRANCH                                      
---------------------------------------------------
 System                = unique
---------------------------------------------------
 self.status           = active
 self.creation_time    = 02/09/2026 17:55:36
 self.creation_user    = svela
 self.path             = /home/svela/SCOPE/Benchmarks/2-Comparison/3-Computations/unique/bench/
 self.name             = bench
 Num Workflows         = 43




In [10]:
# Here we print all jobs that finish successfully, with some relevant data
for wk in branch.workflows:
    for job in wk.jobs:
        if job.isfinished: 
            print(f'Name: {wk.name:8s} job type: {job.name:8s} elapsed time = {job.elapsed_time:8}, ncomputations = {len(job.computations)} charge: {wk.source.charge}')      

Name: ulig_1   job type: scf      elapsed time =     7.55, ncomputations = 1 charge: 0
Name: ulig_1   job type: relax    elapsed time =   273.55, ncomputations = 1 charge: 0
Name: ulig_2   job type: scf      elapsed time =     7.28, ncomputations = 1 charge: 0
Name: ulig_2   job type: relax    elapsed time =  1019.69, ncomputations = 1 charge: 0
Name: ulig_3   job type: scf      elapsed time =      8.4, ncomputations = 1 charge: 0
Name: ulig_3   job type: relax    elapsed time =   274.72, ncomputations = 1 charge: 0
Name: ulig_4   job type: scf      elapsed time =    16.65, ncomputations = 1 charge: -1
Name: ulig_4   job type: relax    elapsed time =   785.35, ncomputations = 1 charge: -1
Name: ulig_5   job type: scf      elapsed time =     3.95, ncomputations = 1 charge: -1
Name: ulig_5   job type: relax    elapsed time =    91.51, ncomputations = 1 charge: -1
Name: ulig_6   job type: scf      elapsed time =    13.52, ncomputations = 1 charge: -2
Name: ulig_6   job type: relax    elap

In [11]:
# Here we print any job that did not finish successfully
print("Unfinished Jobs in Branch:")
for wk in branch.workflows:
    for job in wk.jobs:
        if not job.isfinished: 
            print(f'Workflow name: {wk.name:8s} job type: {job.name:8s} finished = {job.isfinished}, elapsed time = {job.elapsed_time:8}, ncomputations = {len(job.computations)}')      

Unfinished Jobs in Branch:
Workflow name: ulig_11  job type: scf      finished = False, elapsed time =  2696.74, ncomputations = 6
Workflow name: ulig_11  job type: relax    finished = False, elapsed time =      0.0, ncomputations = 0
Workflow name: ulig_25  job type: scf      finished = False, elapsed time =  2081.04, ncomputations = 6
Workflow name: ulig_25  job type: relax    finished = False, elapsed time =      0.0, ncomputations = 0


## Debug

You can use these cells to inspect any individual source. You just need to modify the source name in the cell below

In [12]:
debug_ulig = sys.find_source("ulig_25")[1]

In [13]:
debug_ulig

-----------------------------------
------- SCOPE LIGAND Object -------
-----------------------------------
 Version               = 0.9.6
 Type                  = specie
 Sub-Type              = ligand
 Name                  = ulig_25
 Number of Atoms       = 17
 Formula               = H7-C7-N-O-S
 Charge                = -2
 Spin (alpha - beta)   = 0
 SMILES                = [H]c1c(C([H])([H])[O-])nc(C([H])([H])[S-])c([H])c1[H]
 Number of Parents     = 2
 Has Adjacency Matrix  = YES
 Has Bonds             = YES
 Has RDKIT Object      = YES


In [14]:
debug_ulig.view()